# 📊 Prediksi Volatilitas — XGBoost Teknikal Saja (Baseline)
### Input: Data Teknikal Saja (Tanpa Sentimen)
### Dataset: `dataset_merger_covid2020.csv`

**Fitur X (12 variabel):**
Open, High, Low, Close, Volume, Log\_Return, RSI, MA\_5/10/20, Stoch\_K/D

**Target Y:** Volatilitas → prediksi t+1 (lag features)

## 1. Install & Import

In [ ]:
!pip install -q xgboost scikit-learn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os
warnings.filterwarnings('ignore')
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

EMITEN_LIST = ['ADRO','ADMR','ITMG','PTBA','MEDC','AKRA']
WARNA = {'ADRO':'#6366f1','ADMR':'#10b981','ITMG':'#f59e0b',
         'PTBA':'#ef4444','MEDC':'#8b5cf6','AKRA':'#06b6d4'}

def hitung_metrik(y_true, y_pred, label=''):
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    mask   = y_true != 0
    rmse   = np.sqrt(mean_squared_error(y_true, y_pred))
    mae    = mean_absolute_error(y_true, y_pred)
    mape   = np.mean(np.abs((y_true[mask]-y_pred[mask])/y_true[mask]))*100
    if label:
        print(f'  {label}: RMSE={rmse:.6f} | MAE={mae:.6f} | MAPE={mape:.4f}%')
    return {'RMSE':rmse,'MAE':mae,'MAPE':mape}

from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit

INPUT_CSV   = 'dataset_merger_covid2020.csv'
OUTPUT_DIR  = 'hasil_xgb_teknikal'
os.makedirs(OUTPUT_DIR, exist_ok=True)

FITUR_X = ['Open','High','Low','Close','Volume','Log_Return',
           'RSI','MA_5','MA_10','MA_20','Stoch_K','Stoch_D']
FITUR_Y     = 'Volatilitas'
TRAIN_RATIO = 0.80
N_LAGS      = 5   # tambah lag 1–5 hari sebagai fitur

print(f'✅ Fitur X: {len(FITUR_X)} variabel (tanpa sentimen)')

## 2. Load Data

In [ ]:
df_all = pd.read_csv(INPUT_CSV, encoding='utf-8-sig')
df_all['Tanggal'] = pd.to_datetime(df_all['Tanggal'])
df_all = df_all.sort_values(['Kode_Emiten','Tanggal']).reset_index(drop=True)
print(f'Total: {len(df_all):,} baris | {df_all["Kode_Emiten"].nunique()} emiten')

## 3. Fungsi XGBoost

In [ ]:
def tambah_lag_fitur(df, fitur, n_lag):
    """Tambah fitur lag t-1 s/d t-n untuk menangkap pola temporal."""
    df = df.copy()
    for f in fitur:
        for lag in range(1, n_lag+1):
            df[f'{f}_lag{lag}'] = df[f].shift(lag)
    return df

def tambah_target_tplus1(df, col_y):
    """Buat target t+1 (prediksi hari berikutnya)."""
    df = df.copy()
    df['Y_t1'] = df[col_y].shift(-1)  # volatilitas besok
    return df

print('✅ Fungsi XGBoost siap')

## 4. Training per Emiten

In [ ]:
hasil, models = {}, {}

for kode in EMITEN_LIST:
    print(f'\n{'='*55}')
    print(f'  {kode} — XGBoost Teknikal')
    print(f'{'='*55}')
    df_e = df_all[df_all['Kode_Emiten']==kode].copy()
    df_e = df_e.sort_values('Tanggal').reset_index(drop=True)

    # Tambah lag fitur & target t+1
    df_e = tambah_lag_fitur(df_e, FITUR_X, N_LAGS)
    df_e = tambah_target_tplus1(df_e, FITUR_Y)
    df_e.dropna(inplace=True)

    # Kolom fitur (asli + lag)
    kolom_lag = [f'{f}_lag{i}' for f in FITUR_X for i in range(1,N_LAGS+1)]
    KOLOM_X   = FITUR_X + kolom_lag
    TARGET    = 'Y_t1'

    # Normalisasi
    sc_X = MinMaxScaler(); sc_y = MinMaxScaler()
    X    = sc_X.fit_transform(df_e[KOLOM_X].values)
    y    = sc_y.fit_transform(df_e[[TARGET]].values).flatten()
    tgl  = df_e['Tanggal'].values

    # Split kronologis 80:20
    n_tr = int(len(X)*TRAIN_RATIO)
    X_tr,y_tr = X[:n_tr], y[:n_tr]
    X_te,y_te = X[n_tr:], y[n_tr:]
    tgl_test  = tgl[n_tr:]
    print(f'  Train:{len(X_tr)} | Test:{len(X_te)} | Fitur:{X_tr.shape[1]}')

    # XGBoost
    model = XGBRegressor(
        n_estimators=500, max_depth=5,
        learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, random_state=42,
        early_stopping_rounds=30, eval_metric='rmse',
        verbosity=0,
    )
    model.fit(X_tr,y_tr,eval_set=[(X_te,y_te)],verbose=False)

    # Prediksi
    y_pred = sc_y.inverse_transform(model.predict(X_te).reshape(-1,1)).flatten()
    y_true = sc_y.inverse_transform(y_te.reshape(-1,1)).flatten()

    metrik = hitung_metrik(y_true,y_pred,f'{kode} Test')
    hasil[kode]  = {'metrik':metrik,'y_true':y_true,'y_pred':y_pred,'tgl_test':tgl_test}
    models[kode] = model

print('\n🎉 Training selesai!')

## 5. Rekap & Simpan

In [ ]:
print('\nREKAP EVALUASI — XGBOOST TEKNIKAL')
print(f'  {"Emiten":<8}{"RMSE":>12}{"MAE":>12}{"MAPE":>12}')
rows=[]
for k in EMITEN_LIST:
    if k not in hasil: continue
    m=hasil[k]['metrik']
    print(f'  {k:<8}{m["RMSE"]:>12.6f}{m["MAE"]:>12.6f}{m["MAPE"]:>11.4f}%')
    rows.append({'Emiten':k,**m,'Model':'XGBoost_Teknikal'})

df_eval=pd.DataFrame(rows)
df_eval.to_csv(os.path.join(OUTPUT_DIR,'evaluasi_xgb_teknikal_t1.csv'),
               index=False,encoding='utf-8-sig')

rows_p=[]
for k in EMITEN_LIST:
    if k not in hasil: continue
    h=hasil[k]
    for tgl,akt,pred in zip(h['tgl_test'],h['y_true'],h['y_pred']):
        rows_p.append({'Tanggal':pd.to_datetime(tgl).date(),
                        'Kode_Emiten':k,
                        'Volatilitas_Aktual':round(float(akt),6),
                        'Volatilitas_Pred_XGB_Teknikal_t1':round(float(pred),6),
                        'Error_Abs':round(abs(float(akt)-float(pred)),6)})
pd.DataFrame(rows_p).to_csv(
    os.path.join(OUTPUT_DIR,'hasil_prediksi_xgb_teknikal_t1.csv'),
    index=False,encoding='utf-8-sig')
print('✅ Tersimpan ke hasil_xgb_teknikal/')